# Host Genomes And Viral Regions

Embedded prophages and whole-contig viral candidates are reported separately. These are sequence predictions, not evidence of biological activity. BACPHLIP is assessed only for CheckV-complete Caudoviricetes genomes; incomplete or outside-model-scope sequences remain 'not assessed'. Terminal repeats do not by themselves prove a physically circular or complete genome.

The host summary includes separate CheckV Complete and High-quality region counts. They cover all predicted viral regions (embedded prophages and whole-contig viral candidates). Complete is not counted again as High-quality; regions without a reported CheckV quality do not contribute to either count.

In [ ]:
from __future__ import annotations

import html
import json
import gzip
import math
import os
import re
import shutil
from pathlib import Path

import matplotlib

matplotlib.use("Agg")

import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns


STATUS_ORDER = {"PASS": 0, "OK": 0, "INFO": 0, "WARN": 1, "REVIEW": 1, "FAIL": 2}
CHECKV_GOOD = {"Complete", "High-quality", "Medium-quality"}
CHECKV_COMPLETE = {"Complete", "High-quality"}
CONTIG_SAMPLES = {}
from IPython.display import Image, Markdown, display


In [ ]:
def _setup_plots() -> None:
    sns.set_style("ticks", {"axes.grid": True})
    sns.set_palette("colorblind")
    plt.rcParams["axes.linewidth"] = 1.5
    plt.rcParams["xtick.major.width"] = 1.5
    plt.rcParams["ytick.major.width"] = 1.5
    plt.rcParams["xtick.major.size"] = 8
    plt.rcParams["ytick.major.size"] = 8
    plt.rcParams["axes.titlepad"] = 16
    plt.rcParams["axes.titlesize"] = 18
    plt.rcParams["axes.labelsize"] = 14
    plt.rcParams["xtick.labelsize"] = 10
    plt.rcParams["ytick.labelsize"] = 10
    plt.rcParams["legend.fontsize"] = 10
    plt.rcParams["font.family"] = "sans-serif"
    plt.rcParams["font.sans-serif"] = ["Liberation Sans", "DejaVu Sans"]
    plt.rcParams["text.usetex"] = False
    plt.rcParams["svg.fonttype"] = "none"
    plt.rcParams["savefig.dpi"] = 300


def _as_list(value) -> list:
    if value is None:
        return []
    if isinstance(value, (str, os.PathLike)):
        return [str(value)]
    try:
        return [str(x) for x in value]
    except TypeError:
        return [str(value)]


def _get_named(container, name, default=None):
    try:
        return getattr(container, name)
    except Exception:
        return default


def _ensure_parent(*paths) -> None:
    for path in paths:
        if path:
            Path(str(path)).parent.mkdir(parents=True, exist_ok=True)


def _read_table(path, sep="\t", names=None, header="infer") -> pd.DataFrame:
    path = Path(str(path))
    if not path.exists() or path.stat().st_size == 0:
        return pd.DataFrame(columns=names or [])
    try:
        return pd.read_csv(path, sep=sep, names=names, header=header)
    except pd.errors.EmptyDataError:
        return pd.DataFrame(columns=names or [])


def _to_numeric(df: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    for column in columns:
        if column in df.columns:
            df[column] = pd.to_numeric(df[column], errors="coerce")
    return df


def _first_present(df: pd.DataFrame, candidates: list[str]):
    for candidate in candidates:
        if candidate in df.columns:
            return candidate
    return None


def _read_fasta_lengths(path) -> dict[str, int]:
    path = Path(str(path))
    if not path.exists() or path.stat().st_size == 0:
        return {}
    opener = gzip.open if path.suffix == ".gz" else open
    lengths: dict[str, int] = {}
    current = None
    length = 0
    with opener(path, "rt", errors="replace") as handle:
        for line in handle:
            if line.startswith(">"):
                if current is not None:
                    lengths[current] = length
                current = line[1:].strip().split()[0]
                length = 0
            else:
                length += len(line.strip())
    if current is not None:
        lengths[current] = length
    return lengths



_setup_plots()


In [ ]:
HOSTS=list(snakemake.params.hosts)
checkm=_read_table(snakemake.input.checkm, sep=",").rename(columns={"Bin Id": "host"})
viral_tables=[]
host_rows=[]
for host, fasta, evidence_path, checkv_path, eligibility_path, bacphlip_path, circle_path in zip(
        HOSTS, snakemake.input.host_fastas, snakemake.input.evidence, snakemake.input.checkv,
        snakemake.input.eligibility, snakemake.input.bacphlip, snakemake.input.circularity):
    evidence=_read_table(evidence_path)
    quality=_read_table(checkv_path)
    if "contig_id" not in quality:
        quality["contig_id"]=pd.Series(dtype=object)
    quality=quality.rename(columns={"contig_id": "viral_id"}).add_prefix("CheckV_")
    evidence=evidence.merge(quality, left_on="viral_id", right_on="CheckV_viral_id", how="left")
    eligibility=_read_table(eligibility_path)
    evidence=evidence.merge(eligibility, on="viral_id", how="left")
    bacphlip=_read_table(bacphlip_path)
    if len(bacphlip.columns):
        bacphlip=bacphlip.rename(columns={bacphlip.columns[0]: "viral_id",
                                        "Virulent": "BACPHLIP_virulent", "Temperate": "BACPHLIP_temperate"})
        evidence=evidence.merge(bacphlip, on="viral_id", how="left")
    for field in ["BACPHLIP_virulent", "BACPHLIP_temperate"]:
        if field not in evidence:
            evidence[field]="not assessed"
        evidence[field]=evidence[field].fillna("not assessed")
    circles=_read_table(circle_path).rename(columns={"contig_id": "parent_contig"})
    circle_fields=["parent_contig", "dtr_bp", "itr_bp", "terminal_repeat_type", "repeat_warning"]
    evidence=evidence.merge(circles.reindex(columns=circle_fields), on="parent_contig", how="left")
    # Whole-host-contig terminal repeats are not assigned to an embedded region.
    embedded=evidence["evidence_scope"].eq("embedded_prophage")
    for field in ["dtr_bp", "itr_bp", "terminal_repeat_type", "repeat_warning"]:
        evidence[field]=evidence[field].astype(object)
    evidence.loc[embedded, ["dtr_bp", "itr_bp", "terminal_repeat_type", "repeat_warning"]]="not assessed for extracted region"
    whole=~embedded
    genomad_repeat=evidence["topology"].fillna("").str.strip().str.upper().isin(["DTR", "ITR"])
    # terminal_repeat_type already applies circularity_min_repeat_bp in the producer rule.
    circularity_repeat=evidence["terminal_repeat_type"].fillna("").str.contains(r"(?:^|\+)(?:DTR|ITR)(?:$|\+)", regex=True)
    repeats=genomad_repeat | circularity_repeat
    evidence["whole_contig_terminal_repeat_candidate"]=whole & repeats
    evidence["region_label"]=evidence["evidence_scope"]
    uncertain=whole & evidence.get("CheckV_provirus", pd.Series(index=evidence.index, dtype=str)).eq("Yes")
    evidence.loc[uncertain, "region_label"]="uncertain_viral_contig"
    evidence["BACPHLIP_lifestyle"]="not assessed"
    eligible=evidence["bacphlip_assessment"].eq("eligible")
    predicted=eligible & pd.to_numeric(evidence["BACPHLIP_temperate"], errors="coerce").notna()
    evidence.loc[predicted, "BACPHLIP_lifestyle"]=np.where(
        pd.to_numeric(evidence.loc[predicted, "BACPHLIP_temperate"]) >
        pd.to_numeric(evidence.loc[predicted, "BACPHLIP_virulent"]), "temperate", "virulent")
    viral_tables.append(evidence)
    lengths=_read_fasta_lengths(fasta)
    quality_counts=evidence.get("CheckV_checkv_quality", pd.Series(index=evidence.index, dtype=object)).value_counts()
    host_rows.append(dict(host=host, contigs=len(lengths), genome_length=sum(lengths.values()),
        embedded_prophages=int(embedded.sum()), whole_contig_viral_candidates=int(whole.sum()),
        complete_viral_regions=int(quality_counts.get("Complete", 0)),
        high_quality_viral_regions=int(quality_counts.get("High-quality", 0)),
        whole_contig_terminal_repeat_candidates=int(evidence["whole_contig_terminal_repeat_candidate"].sum())))
hosts=pd.DataFrame(host_rows, columns=["host", "contigs", "genome_length", "embedded_prophages",
                                     "whole_contig_viral_candidates", "complete_viral_regions", "high_quality_viral_regions",
                                     "whole_contig_terminal_repeat_candidates"])
if "host" in checkm:
    hosts=hosts.merge(checkm, on="host", how="left")
for field in ["Completeness", "Contamination", "Strain heterogeneity"]:
    if field not in hosts:
        hosts[field]="not reported"
viral_regions=pd.concat(viral_tables, ignore_index=True) if viral_tables else pd.DataFrame(
    columns=["host", "viral_id", "evidence_scope", "region_label", "length_bp"])
hosts=hosts.sort_values("host", key=lambda values: values.str.casefold()).reset_index(drop=True)
hosts.fillna("not reported").to_csv(snakemake.output.hosts, sep="\t", index=False)
viral_regions.fillna("not reported").to_csv(snakemake.output.viral_regions, sep="\t", index=False)
display(hosts)
display(viral_regions)


## Predicted Viral Regions By Host

Hosts are ordered alphabetically. Hosts with zero predictions remain in the table and plot.

In [ ]:
fig, ax=plt.subplots(figsize=(max(7, len(HOSTS) * 0.7), 5))
if hosts.empty:
    ax.text(0.5, 0.5, "No host FASTA files provided", ha="center", va="center")
    ax.set_axis_off()
else:
    hosts.sort_values("host", key=lambda values: values.str.casefold()).set_index("host")[["embedded_prophages", "whole_contig_viral_candidates"]].plot.bar(
        stacked=True, ax=ax, color=sns.color_palette("colorblind")[:2])
    ax.set_ylabel("geNomad predicted sequences")
    ax.set_xlabel("Host")
    ax.legend(title="Evidence scope")
    sns.despine(ax=ax)
fig.tight_layout()
fig.savefig(snakemake.output.png, dpi=300, bbox_inches="tight")
fig.savefig(snakemake.output.svg, bbox_inches="tight")
plt.close(fig)
display(Image(filename=snakemake.output.png))
parts=["<html><head><meta charset='utf-8'></head><body><h1>Host Genomes And Viral Regions</h1>",
       "<p>Predictions do not establish activity. Whole-contig viral candidates are not labelled embedded prophages. "
       "Terminal-repeat evidence does not establish physical circularity. BACPHLIP is limited to eligible complete phage genomes.</p>",
       hosts.to_html(index=False), f"<img src='{Path(snakemake.output.png).name}' style='max-width:100%'>",
       viral_regions.to_html(index=False), "</body></html>"]
Path(snakemake.output.html).write_text("\n".join(parts))
